## Camada Bronze: Ingestão de Dados Brutos

In [1]:
import pandas as pd
import requests
from google.cloud import bigquery

# Note: Users should ensure PROJECT_ID is set correctly in their environment
# For this automation, we assume the environment is already authenticated.
PROJECT_ID = 'your-project-id' # @param {type:"string"}
DATASET_ID = 'dataset_pratica'
USER_ID = 'user_data_eng'
TABLE_NAME_BRONZE = f'tb_bronze_comments_{USER_ID}'
TABLE_ID_BRONZE = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME_BRONZE}'

# 1. Fetch data from API
print('Fetching data from API...')
url = 'https://dummyjson.com/comments?limit=50'
response = requests.get(url)
data = response.json()['comments']

# 2. Process into DataFrame
df_bronze = pd.DataFrame(data)[['id', 'body', 'likes']]
print(f'Ingested {len(df_bronze)} rows.')
print(df_bronze.head())

Fetching data from API...
Ingested 50 rows.
   id                                       body  likes
0   1             This is some awesome thinking!      3
1   2  What terrific math skills you're showing!      4
2   3                 You are an amazing writer!      2
3   4            Wow! You have improved so much!      1
4   5                                 Nice idea!      1


In [ ]:
# Initialize BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Configure load job
job_config = bigquery.LoadJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

# Load data into BigQuery
job = client.load_table_from_dataframe(df_bronze, TABLE_ID_BRONZE, job_config=job_config)
job.result()  # Wait for the job to complete

print(f'Successfully loaded {len(df_bronze)} rows into {TABLE_ID_BRONZE}')

## Camada Silver: Enriquecimento e Limpeza

In [ ]:
import vertexai
from vertexai.generative_models import GenerativeModel

# Configurações da Camada Silver
TABLE_NAME_SILVER = f'tb_silver_comments_enriched_{USER_ID}'
TABLE_ID_SILVER = f'{PROJECT_ID}.{DATASET_ID}.{TABLE_NAME_SILVER}'

# Inicializar Vertex AI
vertexai.init(project=PROJECT_ID, location='us-central1')
model = GenerativeModel('gemini-2.5-flash')

# 1. Carregar dados da Bronze
query_bronze = f'SELECT * FROM `{TABLE_ID_BRONZE}`'
df_silver = client.query(query_bronze).to_dataframe()

# 2. Função de enriquecimento com LLM
def classificar_comentario(texto):
    prompt = f'Analise o texto e classifique em: Elogio, Crítica, Dúvida ou Sugestão. Responda apenas a palavra: "{texto}"'
    try:
        response = model.generate_content(prompt)
        return response.text.strip()
    except:
        return 'Não Classificado'

print('🤖 Enriquecendo dados com Gemini...')
df_silver['categoria'] = df_silver['body'].apply(classificar_comentario)

# 3. Salvar na tabela Silver
client.load_table_from_dataframe(df_silver, TABLE_ID_SILVER, job_config=job_config).result()
print(f'✅ Camada Silver concluída: {TABLE_ID_SILVER}')
display(df_silver.head())

## Camada Gold: Preparação para Negócio e Visualização

In [ ]:
import plotly.graph_objects as go

# 1. Preparação dos dados para Pareto
df_gold = df_silver['categoria'].value_counts().reset_index()
df_gold.columns = ['categoria', 'quantidade']
df_gold = df_gold.sort_values(by='quantidade', ascending=False)

df_gold['percentual_acumulado'] = (df_gold['quantidade'].cumsum() / df_gold['quantidade'].sum()) * 100

# 2. Visualização
fig = go.Figure()
fig.add_trace(go.Bar(x=df_gold['categoria'], y=df_gold['quantidade'], name='Frequência'))
fig.add_trace(go.Scatter(x=df_gold['categoria'], y=df_gold['percentual_acumulado'], name='% Acumulada', yaxis='y2'))

fig.update_layout(
    title='Análise de Pareto - Feedback de Clientes (Camada Gold)',
    yaxis=dict(title='Quantidade'),
    yaxis2=dict(title='Percentual (%)', overlaying='y', side='right', range=[0, 110]),
    template='plotly_white'
)

fig.show()